# Wearly — Outfit Compatibility Scorer Fine-Tuning

**Model:** Gemma 4 2B (via Unsloth)  
**Module:** `wearly-outfit-v1` — Module 2 (outfit pairing & compatibility scoring)  
**Datasets:** Polyvore Outfits · FashionGen · Synthetic compatibility pairs · Wearly own data  
**Target capabilities:**
- Outfit compatibility scoring (0–100)
- Colour harmony analysis (complementary, analogous, clash)
- Weather & occasion appropriateness rating
- Improvement suggestions per item
- Overall style vibe classification

**Runtime:** Google Colab Pro+ A100 40 GB or local GPU ≥ 24 GB VRAM

---

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install -q unsloth datasets transformers trl peft accelerate bitsandbytes
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
# ── 2. Load base model ────────────────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = 'google/gemma-4-2b-it',
    max_seq_length = MAX_SEQ_LEN,
    dtype         = None,
    load_in_4bit  = True,
)
print('✅ Model loaded')

In [ ]:
# ── 3. LoRA adapters ──────────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha     = 64,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
print('✅ LoRA adapters attached')

In [ ]:
# ── 4. Build training data ────────────────────────────────────────────────────
import json, random
from pathlib import Path
from datasets import load_dataset, Dataset, concatenate_datasets

HF_TOKEN = ''

SYSTEM_PROMPT = """You are Wearly's AI outfit compatibility scorer. Given a list of clothing items
with their colours and categories, evaluate the outfit and return a JSON compatibility report.
Always reply with valid JSON only."""

def make_prompt(instruction: str, inp: str, output: str) -> str:
    user_msg = f"{instruction}\n{inp}" if inp else instruction
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{output}<end_of_turn>"
    )

datasets_to_merge = []

# ── Dataset A: Polyvore-style outfit compatibility ─────────────────────────────
try:
    poly = load_dataset('mteb/polyvore-outfits', split='train', token=HF_TOKEN)
    def fmt_poly(row):
        items    = row.get('items', [])
        compat   = row.get('compatibility', 1)
        score    = int(compat * 100) if isinstance(compat, float) else (85 if compat else 35)
        desc     = ', '.join([i.get('name', 'item') for i in items[:4]]) if items else 'outfit'
        output   = json.dumps({
            'compatibility_score': score,
            'color_harmony': 'analogous' if score > 70 else 'clash',
            'occasion_match': 'casual',
            'weather_fit': 'mild',
            'overall_vibe': 'casual chic' if score > 70 else 'needs work',
            'tips': ['Consider a neutral base' if score < 70 else 'Great colour balance'],
        })
        return {'text': make_prompt('Score the compatibility of this outfit.', f'Outfit: {desc}', output)}
    poly = poly.map(fmt_poly, remove_columns=poly.column_names)
    datasets_to_merge.append(poly)
    print(f'✅ Polyvore Outfits: {len(poly):,} examples')
except Exception as e:
    print(f'⚠️  Polyvore skipped: {e}')

# ── Dataset B: Synthetic high-quality outfit pairs ────────────────────────────
# Manually crafted outfit combinations with expert-level compatibility reasoning
OUTFIT_PAIRS = [
    # (items_desc, score, harmony, occasion, vibe, tips)
    ('Navy blue Oxford shirt, beige chinos, white leather sneakers',
     92, 'analogous', 'smart_casual', 'Clean & Polished',
     ['Perfect neutral base — the navy and beige are a classic pairing', 'White sneakers keep it fresh']),

    ('Black slim jeans, white t-shirt, black leather jacket',
     95, 'monochromatic', 'casual', 'Effortless Cool',
     ['Classic monochrome works every time', 'Add a pop of colour with a watch or bag']),

    ('Olive cargo pants, rust orange hoodie, white sneakers',
     78, 'analogous', 'casual', 'Urban Streetwear',
     ['Earthy tones pair naturally', 'Olive and rust are complementary earth tones']),

    ('Grey trousers, light blue shirt, black leather shoes',
     90, 'complementary', 'office', 'Business Casual',
     ['Grey and light blue is a timeless office combination', 'Black shoes anchor the look']),

    ('Red graphic tee, purple joggers, neon green shoes',
     28, 'clash', 'casual', 'Colour Clash',
     ['Red, purple and neon green clash heavily', 'Swap to neutral joggers (black/grey)', 'Replace neon shoes with white or black']),

    ('Charcoal suit, white dress shirt, burgundy tie',
     96, 'complementary', 'formal', 'Sharp & Sophisticated',
     ['Charcoal and burgundy is a powerful formal combination', 'White shirt provides the perfect contrast']),

    ('Cream linen shirt, light khaki shorts, tan sandals',
     88, 'analogous', 'casual', 'Summer Breeze',
     ['Warm neutrals are cohesive and relaxed', 'Perfect for tropical weather']),

    ('Royal blue blazer, white shirt, navy trousers, brown loafers',
     85, 'tonal', 'smart_casual', 'Navy Power',
     ['Blue family tones work well together', 'Brown loafers add warmth and contrast']),

    ('Mustard yellow tee, black skinny jeans, white sneakers',
     82, 'complementary', 'casual', 'Bold Casual',
     ['Mustard pops against black — a strong complementary pair', 'White sneakers balance the boldness']),

    ('Pink dress shirt, grey trousers, black Oxford shoes',
     87, 'complementary', 'smart_casual', 'Fresh & Confident',
     ['Pink and grey is an underrated classic combination', 'Black shoes keep it grounded']),

    ('Orange shirt, green pants, yellow shoes',
     22, 'clash', 'casual', 'Traffic Light',
     ['Primary colour overload — too many competing hues', 'Keep one bold piece, neutralise the rest', 'Try: white shirt + green pants + brown shoes']),

    ('Sage green linen trousers, white linen shirt, leather sandals',
     91, 'analogous', 'casual', 'Coastal Minimalist',
     ['Sage and white is clean and cohesive', 'Ideal for warm weather and beach destinations']),

    ('Black turtleneck, camel overcoat, dark jeans, Chelsea boots',
     94, 'complementary', 'smart_casual', 'Autumn Edit',
     ['Camel and black is a luxurious combination', 'The monochrome base lets the coat shine']),

    ('Light grey sweatshirt, navy shorts, white socks, white sneakers',
     75, 'analogous', 'casual', 'Sporty Casual',
     ['Cool tones stay cohesive', 'Navy shorts and light grey complement each other']),

    ('Dark wash jeans, burgundy Oxford shirt, brown belt, brown loafers',
     89, 'complementary', 'smart_casual', 'Rich & Refined',
     ['Dark denim and burgundy create depth', 'Brown leather accessories tie the look together']),
]

synthetic_rows = []
for items_desc, score, harmony, occasion, vibe, tips in OUTFIT_PAIRS:
    output = json.dumps({
        'compatibility_score': score,
        'color_harmony': harmony,
        'occasion_match': occasion,
        'weather_fit': 'all-weather' if score > 70 else 'check weather',
        'overall_vibe': vibe,
        'tips': tips,
    })
    synthetic_rows.append({'text': make_prompt(
        'Rate the compatibility of this outfit and explain why.',
        f'Outfit items: {items_desc}.',
        output
    )})

# Weather-specific variants
WEATHER_OUTFITS = [
    ('32°C Singapore heat, 85% humidity',
     'Black wool turtleneck, heavy denim jeans, leather boots',
     18, 'poor', 'Too heavy for weather',
     ['This outfit is dangerously hot for 32°C', 'Switch to linen or cotton in light colours', 'Avoid dark colours that absorb heat']),

    ('32°C Singapore heat, 85% humidity',
     'White linen shirt, light khaki shorts, canvas sneakers',
     96, 'excellent', 'Tropical Perfect',
     ['Light fabrics and colours are ideal for Singapore heat', 'Linen breathes well in humidity']),

    ('10°C London winter, drizzle',
     'Camel wool coat, black turtleneck, dark jeans, Chelsea boots',
     95, 'excellent', 'London Winter Ready',
     ['Layering and warm fabrics are spot on for cold weather', 'Waterproof boots are a smart choice in drizzle']),

    ('10°C London winter, drizzle',
     'Linen shirt, linen trousers, canvas sneakers',
     20, 'poor', 'Too light for conditions',
     ['Linen is inappropriate for 10°C', 'Add a coat, jumper, and waterproof shoes', 'Canvas will soak through immediately in rain']),
]
for weather, outfit, score, weather_fit, vibe, tips in WEATHER_OUTFITS:
    output = json.dumps({
        'compatibility_score': score,
        'color_harmony': 'good' if score > 50 else 'n/a',
        'occasion_match': 'weather-appropriate' if score > 50 else 'weather-inappropriate',
        'weather_fit': weather_fit,
        'overall_vibe': vibe,
        'tips': tips,
    })
    synthetic_rows.append({'text': make_prompt(
        f'Rate this outfit for the weather: {weather}.',
        f'Outfit: {outfit}.',
        output
    )})

# Repeat for better learning
synthetic_rows = synthetic_rows * 80
random.shuffle(synthetic_rows)
synthetic_ds = Dataset.from_list(synthetic_rows)
datasets_to_merge.append(synthetic_ds)
print(f'✅ Synthetic outfit pairs: {len(synthetic_ds):,} examples')

# ── Dataset C: Wearly /api/learn outfit data ───────────────────────────────────
wearly_jsonl = Path('../datasets/wearly_outfit_train.jsonl')
if wearly_jsonl.exists():
    rows = [json.loads(l) for l in open(wearly_jsonl)]
    own_ds = Dataset.from_list([{'text': make_prompt(r.get('instruction','Score this outfit.'), r.get('input',''), r.get('output','{}'))} for r in rows])
    datasets_to_merge.append(own_ds)
    print(f'✅ Wearly own data: {len(own_ds):,} examples')

combined = concatenate_datasets(datasets_to_merge).shuffle(seed=42)
print(f'\n📊 Total training examples: {len(combined):,}')

In [ ]:
# ── 5. Tokenise ───────────────────────────────────────────────────────────────
tokenised = combined.map(
    lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False),
    batched=True,
    remove_columns=['text'],
)
print(f'✅ Tokenised: {len(tokenised):,} examples')

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model      = model,
    tokenizer  = tokenizer,
    train_dataset    = tokenised,
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 4,
    args = TrainingArguments(
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        warmup_steps                 = 60,
        num_train_epochs             = 3,
        learning_rate                = 2e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 20,
        save_steps                   = 200,
        output_dir                   = './wearly-outfit-checkpoints',
        optim                        = 'adamw_8bit',
        weight_decay                 = 0.01,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
    ),
)

print('🚀 Training wearly-outfit-v1 …')
trainer.train()
print('✅ Training complete!')

In [ ]:
# ── 7. Quick eval ─────────────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = (
    f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
    '<start_of_turn>user\n'
    'Rate this outfit for 32°C Singapore weather.\n'
    'Outfit: Navy blue slim chinos (navy), white linen shirt (white), white canvas sneakers.<end_of_turn>\n'
    '<start_of_turn>model\n'
)
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.2, do_sample=True)
print('\n📋 Model output:')
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split('<start_of_turn>model')[-1])

In [ ]:
# ── 8. Save + GGUF export ─────────────────────────────────────────────────────
model.save_pretrained('wearly-outfit-v1')
tokenizer.save_pretrained('wearly-outfit-v1')

model.save_pretrained_gguf(
    'wearly-outfit-v1-gguf',
    tokenizer,
    quantization_method = 'q4_k_m',
)
print('✅ GGUF saved → wearly-outfit-v1-gguf/')

In [ ]:
# ── 9. Push to Hub (optional) ─────────────────────────────────────────────────
HF_REPO = 'harsaikron/wearly-outfit-v1'

if HF_TOKEN:
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f'✅ Pushed → https://huggingface.co/{HF_REPO}')
else:
    print('⏭  HF_TOKEN not set — skipping')

## ✅ Register with Ollama (run on your Mac)

```bash
# 1. Create Modelfile
cat > ~/Modelfile.outfit << 'EOF'
FROM ~/wearly-outfit-v1-gguf/unsloth.Q4_K_M.gguf
SYSTEM """You are Wearly's AI outfit compatibility scorer. Given a list of clothing items
with their colours and categories, evaluate the outfit and return a JSON report with:
compatibility_score (0-100), color_harmony, occasion_match, weather_fit, overall_vibe, tips (array).
Return valid JSON only."""
PARAMETER temperature 0.2
PARAMETER num_ctx 4096
PARAMETER stop "<end_of_turn>"
EOF

# 2. Register
ollama create wearly-outfit-v1 -f ~/Modelfile.outfit

# 3. Test
ollama run wearly-outfit-v1 "Rate: Navy chinos, white linen shirt, white sneakers. Weather: 32°C Singapore."

# 4. Verify
curl http://localhost:11434/api/tags | jq '.models[].name'
# Should include: wearly-outfit-v1
```

### Scoring guide

| Score | Label | Meaning |
|---|---|---|
| 90–100 | ⭐ Perfect | Textbook combination |
| 75–89  | ✅ Good | Works well, minor tweaks possible |
| 55–74  | ⚠️ Okay | Functional but improvable |
| 30–54  | ⚠️ Needs work | Colour or occasion mismatch |
| 0–29   | ❌ Clash | Major issues — redesign the outfit |